# Feature engineering: from 2,508 candidates to 700 predictors

**Research question:** which information about an applicant's existing credit,
payment and application history survives a future-period validation test?

The frozen feature engine builds 34 train/test blocks from 17 relational table
groups. Numeric distributions, application-relative dates, five recency windows,
categorical diversity, missingness and applicant/related-person subgroups produce
a broad candidate space. These are case-local features; frequency encoders learn
their mappings only on each model's training population.

This notebook audits the original experiment. It does not rerun screening or use
the now-observed final holdout to change the released model.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from home_credit.modeling.portfolio import load_portfolio

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
evidence = load_portfolio(root)
print("Verified immutable experiment evidence; no model fitting.")

features = evidence["features"]
display(
    pd.DataFrame(
        [
            {
                "Candidates": features["candidate_count"],
                "Structurally eligible": features["eligible_count"],
                "Retained": features["retained_count"],
                "Rejected": features["rejected_count"],
            }
        ]
    )
)
display(pd.Series(features["decisions"], name="Count").to_frame())

## Where the information comes from

Credit-bureau records describe overdue payments, balances and historical loan
behavior. Previous applications describe earlier affordability and credit demand.
Person tables distinguish the applicant from related people; tax, deposit and
debit-card tables contribute additional historical context. The application and
bureau snapshots supply current-at-application summaries.

Dates become offsets from `date_decision`; counts use the previous 30, 180, 365,
730 and 1,825 days. Group-index first/last features describe source ordering,
**not a verified chronological trend**. Availability relies on the competition's
application-time snapshots, not a production event-time lineage guarantee.

In [ ]:
from home_credit.modeling.portfolio_report import display_charts

display(pd.DataFrame(features["families"]).fillna(0).set_index("family"))
display_charts(evidence, "features")

## Training-only selection and its limits

Screening fits on weeks 0-24, uses weeks 25-32 for early stopping and temporal
drift diagnostics, and finishes before the five model-validation folds begin
at week 33. It rejects extreme missingness, constants and high-cardinality
categoricals, then ranks training-window predictive gain with a drift penalty.

The 1,334 eligible features below the 700-feature limit are **not proven useless**.
The limit is a computation budget; no 700-versus-larger-set ablation was run.
The near-perfect early-versus-later drift classifier makes temporal testing essential.
Feature gain is associative and is not a causal explanation or a substitute for ablation.

In [ ]:
catalog = pd.DataFrame(features["catalog"])
columns = ["name", "family", "target_gain", "drift_gain", "selection_score", "decision"]
ranked = catalog[catalog["selected"]].sort_values("selection_score", ascending=False)
display(ranked[columns].head(15))
print("Temporal drift-classifier AUC:", round(features["drift_validation_auc"], 6))

## What the ablations establish

Under the same LightGBM control and five expanding folds, removing credit bureau A,
previous applications or depth-two history reduced mean official stability.
This supports retaining those families in this representation. It does not prove
that every individual retained column helps, or that all possible feature families
have been exhausted. The detailed executed ablation is notebook 06.

In [ ]:
display(
    pd.DataFrame(evidence["ablation"]["rows"])[
        ["experiment", "mean_fold_stability", "delta_vs_control", "worst_fold_stability", "oof_auc"]
    ].round(6)
)

## Explicit research boundary

The frozen release does not include custom affordability ratios, cross-table
interactions, quantile/skew features, peer ranks, target encoding, or exhaustive
redundancy and feature-limit ablations. There is no completed SHAP or permutation
stability study. These remain extensions, not completed experiments or claims of
exhaustive discovery. External data and target encoding are unnecessary for the
released pipeline and would require their own availability and leakage controls.

The full aggregate feature catalog is `reports/feature_ablation/feature_screen.json`;
each entry records provenance, dtype, missingness, cardinality and both model gains.
Rejection reasons here are reconstructed from that immutable record and its original
rules. The feature/model lineage is checked before any results are displayed.